# Step 12 — ERA5 MJO Download (u850, u200, OLR)
**Project:** ENSO-BSISO Self-Supervised Learning — MJO Extension  
**Author:** Jiayi (jh9141@nyu.edu)

Downloads ERA5 fields needed for the Wheeler & Hendon (2004) RMM preprocessing pipeline:
- `u850` — zonal wind at 850 hPa  
- `u200` — zonal wind at 200 hPa  
- `OLR`  — top net thermal radiation (`ttr`)

**Key differences from the BSISO download (nb01b):**
| Aspect | BSISO (nb01b) | MJO (this notebook) |
|--------|--------------|--------------------|
| Wind variables | u850 + **v850** | u850 + **u200** |
| Domain | 60°E–160°E, 0°–60°N | **Global, 15°S–15°N** |
| Months | MJJAS | **All 12 months** (all-year) |
| N days | ~6,600 | **~16,425** |

**Domain:** 15°S–15°N, all longitudes (−180° to 180°), 2° resolution  
**Period:** All months, 1979–2023  
**Output files (Google Drive → `BSISO_SSL_Project/MJO/data/raw/`):**
```
u850_u200_1979_1988.nc   ← both pressure levels in one file, 5 year-chunks
u850_u200_1989_1998.nc
u850_u200_1999_2008.nc
u850_u200_2009_2018.nc
u850_u200_2019_2023.nc
OLR_MJO_1979_2023.nc     ← single file, all years
```

**Estimated download time:** 40–80 min total (CDS queue dependent)  
**Estimated file sizes:** ~3–6 MB per wind chunk; ~80–120 MB for OLR

---
⚠️ **Prerequisite:** CDS API key from https://cds.climate.copernicus.eu/ (same account as BSISO project)

## Cell 1 — Mount Google Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_RAW_DIR = f'{PROJECT_DIR}/MJO/data/raw'

os.makedirs(MJO_RAW_DIR, exist_ok=True)

print('Google Drive mounted.')
print(f'MJO raw data folder: {MJO_RAW_DIR}')
print('Files currently in MJO/data/raw/:')
for f in sorted(os.listdir(MJO_RAW_DIR)):
    mb = os.path.getsize(f'{MJO_RAW_DIR}/{f}') / 1e6
    print(f'  {f}  ({mb:.1f} MB)')

## Cell 2 — Install CDS API Client

In [ ]:
!pip install cdsapi --quiet
import cdsapi
print('cdsapi ready.')

## Cell 3 — Set Up CDS API Credentials

In [ ]:
# ============================================================
# FILL IN YOUR PERSONAL ACCESS TOKEN HERE
# Find it at: https://cds.climate.copernicus.eu/ → Your profile
# ============================================================
CDS_API_KEY = 'YOUR_CDS_API_KEY_HERE'
# ============================================================

cdsapirc = f'url: https://cds.climate.copernicus.eu/api\nkey: {CDS_API_KEY}\n'
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(cdsapirc)

print('CDS credentials saved.')

try:
    client = cdsapi.Client(quiet=True)
    print('CDS API connection: OK')
except Exception as e:
    print(f'CDS API connection FAILED: {e}')

## Cell 4 — Download u850 + u200 (1-Year Chunks)\n\nBoth pressure levels (850 hPa and 200 hPa) are downloaded together in a single CDS request per year.  \nCDS limits requests to ~1,000 fields; 1 year × 12 months × 2 levels × ~365 days ≈ 730 fields — safely within limits.  \nDomain: **global equatorial strip 15°S–15°N**  \nGrid: 2° × 2°\n\n⚠️ **45 annual chunks** — run overnight or in batches. Already-downloaded files are skipped automatically."

In [ ]:
import cdsapi
import os
import xarray as xr

if not os.path.exists(os.path.expanduser('~/.cdsapirc')):
    raise RuntimeError('Run Cell 3 first to set up CDS credentials.')

client = cdsapi.Client()

ALL_MONTHS = [f'{m:02d}' for m in range(1, 13)]
DAYS       = [f'{d:02d}' for d in range(1, 32)]  # CDS ignores invalid dates

# DAILY AVERAGE (replaces the old single 12:00 UTC snapshot).
# Winds are INSTANTANEOUS -> download 4x/day (00,06,12,18 UTC) and average to a daily MEAN.
# 1 yr x 365 d x 2 levels x 4 times = ~2,920 fields/chunk, within CDS limits.
INST_TIMES = ['00:00', '06:00', '12:00', '18:00']

def _aggregate_daily(sub_file, out_file, keep_vars, how):
    """Collapse a sub-daily ERA5 file to one value per calendar day.
    how='mean' (instantaneous fields) or 'sum' (accumulated fields). Empty day-bins become NaN
    and are dropped; aux coords (number/expver) stripped before reducing. The pressure_level
    dimension is preserved."""
    ds = xr.open_dataset(sub_file)
    tdim = 'valid_time' if 'valid_time' in ds.dims else 'time'
    ds = ds[keep_vars].reset_coords(drop=True)
    rs = ds.resample(**{tdim: '1D'})
    daily = rs.mean() if how == 'mean' else rs.sum(min_count=1)
    daily = daily.dropna(dim=tdim, how='all')
    daily.to_netcdf(out_file)
    ds.close()
    os.remove(sub_file)

years = list(range(1979, 2024))
print(f'{len(years)} annual chunks to download. Already-existing files are skipped.')

for yr in years:
    out_file = f'{MJO_RAW_DIR}/u850_u200_{yr}.nc'

    if os.path.exists(out_file):
        size_mb = os.path.getsize(out_file) / 1e6
        print(f'[SKIP] u850_u200_{yr}.nc  ({size_mb:.1f} MB)')
        continue

    sub_file = out_file.replace('.nc', '_subdaily.nc')
    print(f'Downloading u850+u200  {yr}  all months (4x/day) ...', end=' ', flush=True)

    client.retrieve(
        'reanalysis-era5-pressure-levels',
        {
            'product_type'  : 'reanalysis',
            'variable'      : 'u_component_of_wind',
            'pressure_level': ['850', '200'],
            'year'          : str(yr),
            'month'         : ALL_MONTHS,
            'day'           : DAYS,
            'time'          : INST_TIMES,
            'area'          : [15, -180, -15, 180],    # N, W, S, E — global 15°S-15°N
            'grid'          : [2.0, 2.0],
            'data_format'   : 'netcdf',
        },
        sub_file
    )

    # Average the 4 daily snapshots -> daily mean; write final file; remove sub-daily temp
    _aggregate_daily(sub_file, out_file, keep_vars=['u'], how='mean')

    size_mb = os.path.getsize(out_file) / 1e6
    print(f'done  ({size_mb:.1f} MB)')

print('\nAll wind chunks done.')

## Cell 5 — Download OLR (1-Year Chunks)\n\nSame variable (`top_net_thermal_radiation`) as BSISO project but over the global equatorial strip.  \nOne year per request (~365 fields), matching the wind chunk strategy."

In [ ]:
import xarray as xr
years = list(range(1979, 2024))
print(f'{len(years)} annual OLR chunks to download. Already-existing files are skipped.')

# OLR (top net thermal radiation) is ACCUMULATED -> download all 24 hourly steps and SUM to a
# full daily accumulation (replaces the old 12:00 snapshot). A full year x 24 h over the global
# 15S-15N strip is large enough to risk the CDS 403 'cost limits exceeded', so each YEAR is split
# into TWO half-year sub-requests (months 1-6 and 7-12), aggregated to daily and concatenated
# into the annual file. Magnitude washes out downstream (anomaly removal + std-norm in nb13).
ACC_TIMES    = [f'{h:02d}:00' for h in range(24)]
MONTH_HALVES = [['01', '02', '03', '04', '05', '06'], ['07', '08', '09', '10', '11', '12']]

def _daily_from_subdaily(ds, keep_vars, how):
    tdim = 'valid_time' if 'valid_time' in ds.dims else 'time'
    ds = ds[keep_vars].reset_coords(drop=True)
    rs = ds.resample(**{tdim: '1D'})
    daily = rs.mean() if how == 'mean' else rs.sum(min_count=1)
    return daily.dropna(dim=tdim, how='all')

for yr in years:
    out_file = f'{MJO_RAW_DIR}/OLR_MJO_{yr}.nc'

    if os.path.exists(out_file):
        size_mb = os.path.getsize(out_file) / 1e6
        print(f'[SKIP] OLR_MJO_{yr}.nc  ({size_mb:.1f} MB)')
        continue

    daily_parts = []
    for h, months in enumerate(MONTH_HALVES):
        sub_h = f'{MJO_RAW_DIR}/_olr_sub_{yr}_h{h}.nc'
        print(f'Downloading OLR {yr} months {months[0]}-{months[-1]} (24x/day) ...', end=' ', flush=True)
        client.retrieve(
            'reanalysis-era5-single-levels',
            {
                'product_type': 'reanalysis',
                'variable'    : 'top_net_thermal_radiation',
                'year'        : str(yr),
                'month'       : months,
                'day'         : DAYS,
                'time'        : ACC_TIMES,
                'area'        : [15, -180, -15, 180],
                'grid'        : [2.0, 2.0],
                'data_format' : 'netcdf',
            },
            sub_h
        )
        ds = xr.open_dataset(sub_h)
        daily_parts.append(_daily_from_subdaily(ds, ['ttr'], 'sum').load())
        ds.close(); os.remove(sub_h); print('done')

    tdim = 'valid_time' if 'valid_time' in daily_parts[0].dims else 'time'
    olr_daily = xr.concat(daily_parts, dim=tdim).sortby(tdim)
    olr_daily.to_netcdf(out_file)
    size_mb = os.path.getsize(out_file) / 1e6
    print(f'  saved OLR_MJO_{yr}.nc  ({size_mb:.1f} MB, {olr_daily.sizes[tdim]} days)')

print('\nAll OLR chunks done.')

## Cell 6 — Verify Downloads

In [ ]:
import os
import xarray as xr
import numpy as np
import pandas as pd

# --- self-contained: works even if Cell 1 wasn't run this session ---
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_RAW_DIR = f'{PROJECT_DIR}/MJO/data/raw'

print('=' * 65)
print('VERIFICATION REPORT — MJO ERA5 Downloads')
print('=' * 65)

# --- Wind ---
print('\n[1] Wind files (u850 + u200)')
wind_files = sorted([
    f'{MJO_RAW_DIR}/{f}' for f in os.listdir(MJO_RAW_DIR)
    if f.startswith('u850_u200') and f.endswith('.nc')
])
print(f'  Found {len(wind_files)} annual files  (expected 45)')

ds_wind_all = xr.concat([xr.open_dataset(f) for f in wind_files],
                         dim='valid_time').sortby('valid_time')
total_days = len(ds_wind_all.valid_time)
n_lat      = len(ds_wind_all.latitude)
n_lon      = len(ds_wind_all.longitude)
lats       = ds_wind_all.latitude.values
lons       = ds_wind_all.longitude.values

print(f'  Combined: {total_days} time steps  (expected ~16,425)')
print(f'  Grid:     {n_lat} lat × {n_lon} lon')
print(f'  Levels:   {sorted(ds_wind_all.pressure_level.values.tolist())} hPa')
months_wind = sorted(set(pd.DatetimeIndex(ds_wind_all.valid_time.values).month))
print(f'  Months:   {months_wind}  (expected all 12)')

u850_vals = ds_wind_all['u'].sel(pressure_level=850).values
u200_vals = ds_wind_all['u'].sel(pressure_level=200).values
print(f'  u850 range: [{u850_vals.min():.1f}, {u850_vals.max():.1f}] m/s')
print(f'  u200 range: [{u200_vals.min():.1f}, {u200_vals.max():.1f}] m/s')

# --- OLR ---
print('\n[2] OLR files')
olr_files = sorted([
    f'{MJO_RAW_DIR}/{f}' for f in os.listdir(MJO_RAW_DIR)
    if f.startswith('OLR_MJO_') and f.endswith('.nc')
])
print(f'  Found {len(olr_files)} annual files  (expected 45)')

ds_olr = xr.concat([xr.open_dataset(f) for f in olr_files],
                    dim='valid_time').sortby('valid_time')
n_olr  = len(ds_olr.valid_time)
t_min  = str(ds_olr.valid_time.values[0])[:10]
t_max  = str(ds_olr.valid_time.values[-1])[:10]
print(f'  Combined: {n_olr} days  {t_min} → {t_max}')
months_olr = sorted(set(pd.DatetimeIndex(ds_olr.valid_time.values).month))
print(f'  Months:   {months_olr}  (expected all 12)')

# --- NaN ---
nan_u850 = int(np.isnan(u850_vals).sum())
nan_u200 = int(np.isnan(u200_vals).sum())
nan_olr  = int(np.isnan(ds_olr['ttr'].values).sum())
print(f'\n[3] NaN counts')
print(f'  u850: {nan_u850}  u200: {nan_u200}  OLR: {nan_olr}  (all expected 0)')

# --- Date alignment ---
wind_dates = set(pd.DatetimeIndex(ds_wind_all.valid_time.values).normalize())
olr_dates  = set(pd.DatetimeIndex(ds_olr.valid_time.values).normalize())
diff_wo    = len(wind_dates - olr_dates)
diff_ow    = len(olr_dates  - wind_dates)
print(f'\n[4] Date alignment')
if diff_wo == 0 and diff_ow == 0:
    print('  OK: wind and OLR cover identical dates')
else:
    print(f'  WARNING: wind-only={diff_wo}, OLR-only={diff_ow}')

# --- Per-year OLR coverage (so a partial resume is obvious) ---
print(f'\n[5] OLR year coverage')
olr_years = sorted(set(pd.DatetimeIndex(ds_olr.valid_time.values).year))
missing = [y for y in range(1979, 2024) if y not in olr_years]
print(f'  OLR years present: {olr_years[0]}–{olr_years[-1]}  ({len(olr_years)}/45)')
if missing:
    print(f'  STILL MISSING: {missing}  -> rerun Cell 5 to fetch these')
else:
    print('  All 45 OLR years present.')

print('\nVerification complete.')

## Cell 7 — Quick Plot (Visual Sanity Check)

Plots a boreal winter (January) and boreal summer (July) day to verify the known MJO-band climatology:  
- **u200** should show strong subtropical jet (westerlies) at higher latitudes, weaker at equator  
- **u850** shows low-level easterlies in tropics year-round  
- **OLR** should show equatorial convection concentrated over the warm pool (Indian Ocean/West Pacific)

In [ ]:
import matplotlib.pyplot as plt

# ds_wind_all and ds_olr are already loaded from Cell 6
times_wind = pd.DatetimeIndex(ds_wind_all.valid_time.values)
times_olr  = pd.DatetimeIndex(ds_olr.valid_time.values)

idx_jan_w = int(np.where(times_wind.month == 1)[0][0])
idx_jul_w = int(np.where(times_wind.month == 7)[0][0])
idx_jan_o = int(np.where(times_olr.month  == 1)[0][0])
idx_jul_o = int(np.where(times_olr.month  == 7)[0][0])

u850_jan = ds_wind_all['u'].isel(valid_time=idx_jan_w).sel(pressure_level=850).values
u850_jul = ds_wind_all['u'].isel(valid_time=idx_jul_w).sel(pressure_level=850).values
u200_jan = ds_wind_all['u'].isel(valid_time=idx_jan_w).sel(pressure_level=200).values
u200_jul = ds_wind_all['u'].isel(valid_time=idx_jul_w).sel(pressure_level=200).values
olr_jan  = -ds_olr['ttr'].isel(valid_time=idx_jan_o).values
olr_jul  = -ds_olr['ttr'].isel(valid_time=idx_jul_o).values

date_jan_w = str(times_wind[idx_jan_w])[:10]
date_jul_w = str(times_wind[idx_jul_w])[:10]
date_jan_o = str(times_olr[idx_jan_o])[:10]
date_jul_o = str(times_olr[idx_jul_o])[:10]

fig, axes = plt.subplots(3, 2, figsize=(18, 11))
fig.suptitle('MJO ERA5 Sanity Check — global equatorial strip (15°S–15°N)', fontsize=14, fontweight='bold')

panels = [
    (u850_jan, f'u850  Jan ({date_jan_w})', 'RdBu_r', 'm/s'),
    (u850_jul, f'u850  Jul ({date_jul_w})', 'RdBu_r', 'm/s'),
    (u200_jan, f'u200  Jan ({date_jan_w})', 'RdBu_r', 'm/s'),
    (u200_jul, f'u200  Jul ({date_jul_w})', 'RdBu_r', 'm/s'),
    (olr_jan,  f'OLR   Jan ({date_jan_o})', 'RdBu_r', 'J/m²'),
    (olr_jul,  f'OLR   Jul ({date_jul_o})', 'RdBu_r', 'J/m²'),
]

for ax, (data, title, cmap, unit) in zip(axes.flat, panels):
    im = ax.imshow(data, cmap=cmap, origin='upper',
                   extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                   aspect='auto')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
    plt.colorbar(im, ax=ax, label=unit, shrink=0.8)

plt.tight_layout()
fig_path = f'{PROJECT_DIR}/MJO/results/era5_mjo_sanity_check.png'
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print()
print('Expected patterns:')
print('  u850: low-level easterlies (negative) over most of tropics')
print('  u200: upper-level westerlies (positive) over tropics, reversed from u850')
print('  OLR:  strong convection (large negative ttr = positive OLR) over warm pool')

---\n## Done!\n\nGoogle Drive should now contain:\n\n```\nBSISO_SSL_Project/MJO/data/raw/\n├── rmm_labels.csv              ← from nb11\n├── nino34_monthly.txt          ← from nb11\n├── rmm_raw.txt                 ← from nb11\n├── u850_u200_1979.nc           ← from this notebook ✓  (45 annual files)\n├── u850_u200_1980.nc\n├── ...\n├── u850_u200_2023.nc\n├── OLR_MJO_1979.nc             ← from this notebook ✓  (45 annual files)\n├── OLR_MJO_1980.nc\n├── ...\n└── OLR_MJO_2023.nc\n```\n\n**Next step:** Run `13_mjo_preprocessing.ipynb`  \nRequires both nb11 (`rmm_labels.csv`) and nb12 (all ERA5 files) to be complete.\n\n---\n*DDCS Project | jh9141@nyu.edu*"